# Exercise 1 — log_result and format_log_entry

Logging is the safety net for any automated system. Every time the bot runs, it appends a timestamped entry to a log file. If the bot misbehaves in production, the log tells you exactly what happened and when. `format_log_entry` adds the timestamp; `log_result` handles file I/O with automatic directory creation.

In [ ]:
import pandas as pd, math, datetime, pathlib, tempfile

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def format_log_entry(report_text):
    """Prepend a timestamp header to report_text.

    Format:
        ====...====
        [2025-01-15 16:00:00]
        ====...====
        <report_text>

    Returns:
        str
    """
    # TODO: ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    # then build the header string and prepend to report_text
    return report_text


def log_result(report_text, path):
    """Append a timestamped report entry to a log file.

    Steps:
      1. path = pathlib.Path(path)
      2. path.parent.mkdir(parents=True, exist_ok=True)
      3. entry = format_log_entry(report_text)
      4. open(path, "a") and write entry + "\n"
    """
    # TODO: ~5 lines
    pass


### Checks

In [ ]:
checks = 0
sample = "=== Paper Trading Report ===\nTotal return: 5.00%"

# 1 — format_log_entry returns a string containing the original text
try:
    entry = format_log_entry(sample)
    assert isinstance(entry, str)
    assert sample in entry, "format_log_entry should include the original report text"
    checks += 1; print("✅ 1 format_log_entry includes the original report text")
except Exception as e:
    print("❌ 1:", e)

# 2 — format_log_entry includes a timestamp-like pattern
try:
    entry = format_log_entry(sample)
    import re
    has_ts = bool(re.search(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", entry))
    assert has_ts, f"no YYYY-MM-DD HH:MM:SS pattern found in:\n{entry}"
    checks += 1; print("✅ 2 format_log_entry includes a YYYY-MM-DD HH:MM:SS timestamp")
except Exception as e:
    print("❌ 2:", e)

# 3 — log_result creates the file
try:
    with tempfile.TemporaryDirectory() as td:
        p = pathlib.Path(td) / "subdir" / "bot.log"
        log_result(sample, p)
        assert p.exists(), "log file should be created"
    checks += 1; print("✅ 3 log_result creates the file (and parent dirs)")
except Exception as e:
    print("❌ 3:", e)

# 4 — log_result content is readable and contains the report
try:
    with tempfile.TemporaryDirectory() as td:
        p = pathlib.Path(td) / "bot.log"
        log_result(sample, p)
        content = p.read_text(encoding="utf-8")
        assert "Paper Trading Report" in content
        assert "5.00%" in content
    checks += 1; print("✅ 4 log file contains the report text")
except Exception as e:
    print("❌ 4:", e)

# 5 — log_result appends (multiple calls grow the file)
try:
    with tempfile.TemporaryDirectory() as td:
        p = pathlib.Path(td) / "bot.log"
        log_result("Entry 1", p)
        log_result("Entry 2", p)
        content = p.read_text(encoding="utf-8")
        assert "Entry 1" in content and "Entry 2" in content
        assert content.count("Entry") == 2
    checks += 1; print("✅ 5 log_result appends — both entries appear in file")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
